<a href="https://colab.research.google.com/github/mahendrapd/GRST/blob/main/DoS_RST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/dataset/training.parquet')

In [ ]:
cols=len(data.columns)
nfeatures = cols-1
X = data.iloc[:,0:nfeatures]
y = data.iloc[:,-1]
print(len(data),cols)

347122 55


In [ ]:
targetclass='DoS'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [ ]:
def RSTClassifier(X,y,targetclass=y[0]):
    #print(targetclass)
    dc=pd.Series(y)
    cm=dc.groupby(dc).apply(lambda px: px.index.tolist()).to_dict() #index of targetclass
    D=set(cm[targetclass])
    # D1=set(cm['Bruteforce'])
    # D2=set(cm['Botnet'])
    # D3=set(cm['Portscan'])
    # D4=set(cm['Webattack'])
    # D5=set(cm['DoS'])
    # D6=set(cm['Infiltration'])
    # D7=set(cm['DDoS'])
    print(D)
    nfeatures=len(X.columns)
    #print(cm)
    purity={}
    headers = X.keys().tolist()

    for j in range(nfeatures):
        header=headers[j]
        hdata={}
        #print("Columns:",j,header)
        dp=pd.Series(X.iloc[:,j])
        md=dp.groupby(dp).apply(lambda px: px.index.tolist()).to_dict()
        K=md.keys()
        #print(K)
        for z in K:
            common_elements=set(md[z]) & D
            val=round(len(common_elements)/len(md[z]),3)
            #print(z,len(md[z]),len(common_elements),val)
            hdata[z]=val
        purity[header]=hdata
    #print(purity['urgent'][0])
    return purity

In [ ]:
pur=RSTClassifier(X_train,y_train,targetclass)

{196621, 196623, 196624, 196626, 196632, 196633, 196634, 196635, 196636, 196637, 196638, 196640, 196641, 196643, 196644, 196645, 196647, 196649, 196651, 196652, 196654, 196655, 196660, 196662, 196663, 196667, 196669, 196673, 196674, 196675, 196676, 196678, 196679, 196683, 196684, 196686, 196691, 196695, 196696, 196698, 196699, 196701, 196702, 196703, 196724, 196725, 196727, 196729, 196731, 196733, 196734, 196735, 196739, 196740, 196743, 196747, 196748, 196751, 196752, 196754, 196755, 196758, 196763, 196765, 196766, 196770, 196771, 196772, 196774, 196776, 196777, 196778, 196779, 196780, 196781, 196782, 196786, 196787, 196788, 196789, 196792, 196794, 196795, 196796, 196798, 196800, 196801, 196802, 196803, 196804, 196805, 196806, 196807, 196811, 196812, 196813, 196823, 196828, 196829, 196830, 196831, 196832, 196833, 196835, 196838, 196839, 196843, 196844, 196845, 196847, 196848, 196849, 196851, 196856, 196858, 196859, 196860, 196866, 196868, 196869, 196871, 196872, 196873, 196874, 196875,

In [ ]:
#print(pur)

In [ ]:
def RST_AmbigiousSample(X,pur):
    headers = X.keys().tolist()
    print(headers,len(headers),len(X))
    ambigioussamples={}
    for i in range(len(X)):
        flag=0
        sumpurity=0
        for j in range(len(headers)):
            key=X.iloc[i][headers[j]]
            if (pur[headers[j]][key] == 1 or pur[headers[j]][key] == 0):
                flag=1
                break
            else:
                sumpurity+=pur[headers[j]][key]
        if (flag == 0):
            Avgpurity=round(sumpurity/len(headers),3)
            ambigioussamples[i]=Avgpurity
    return ambigioussamples

In [ ]:
AmbSample=RST_AmbigiousSample(X_train,pur)

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle 

In [ ]:
print(len(AmbSample))

50013


In [ ]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [ ]:
def RST_Gradient(y,sample,epoch=300):
    Theta=0.5
    LRate=0.01
    pc=0
    for i in range(epoch):
        correctcount=0
        val=0
        Th=Theta
        L=0

        for key, value in sample.items():
            if (value >= Theta and y.iloc[key] == targetclass):
                correctcount+=1
            elif (value < Theta and y.iloc[key] != targetclass):
                correctcount+=1
            else:
                val+=value
        ictotal=(len(sample)-correctcount)
        try:
          p=correctcount/len(sample)
          #L=-(Th*math.log2(p)+(1-Th)*math.log2(1-p))
          #L=-((Th/p)-((1-Th)/(1-p)))
          #L=np.tanh(L)
          #Theta=abs(round((Theta-LRate*L),3))
          #print(L)
          if (pc<=correctcount):
            Theta=abs(round(Theta-LRate*(sigmoid((val/ictotal))),3))
            pc=correctcount
          #Theta=round(Theta-LRate*(0-(val/ictotal))*(val/ictotal)*(1-(val/ictotal)),3)
        except:
          Theta=Theta
        print(i, correctcount, ictotal, len(sample), val, Theta)

    return Theta

In [ ]:
Th=RST_Gradient(y_train,AmbSample,300)

0 41975 8038 50013 683.8270000000015 0.495
1 41975 8038 50013 683.8270000000015 0.49
2 41975 8038 50013 683.8270000000015 0.485
3 41975 8038 50013 683.8270000000015 0.48
4 41975 8038 50013 683.8270000000015 0.475
5 41975 8038 50013 683.8270000000015 0.47
6 41975 8038 50013 683.8270000000015 0.465
7 41975 8038 50013 683.8270000000015 0.46
8 41975 8038 50013 683.8270000000015 0.455
9 41975 8038 50013 683.8270000000015 0.45
10 41975 8038 50013 683.8270000000015 0.445
11 41975 8038 50013 683.8270000000015 0.44
12 41975 8038 50013 683.8270000000015 0.435
13 41975 8038 50013 683.8270000000015 0.43
14 41975 8038 50013 683.8270000000015 0.425
15 41975 8038 50013 683.8270000000015 0.42
16 41975 8038 50013 683.8270000000015 0.415
17 41975 8038 50013 683.8270000000015 0.41
18 41975 8038 50013 683.8270000000015 0.405
19 41975 8038 50013 683.8270000000015 0.4
20 41975 8038 50013 683.8270000000015 0.395
21 41975 8038 50013 683.8270000000015 0.39
22 41975 8038 50013 683.8270000000015 0.385
23 41975 8

In [ ]:
def RST_Validation(X,y,pur,Theta):
    TP=0
    TN=0
    FP=0
    FN=0
    headers = X.keys().tolist()
    for i in range(len(X)):
        val=0
        count=0
        pval=0

        flag=0
        for j in range(len(headers)):
            #KYS=list(pur[headers[j]].keys())
            #dflag=0
            key=X.iloc[i][headers[j]]
            try:
              pval=pur[headers[j]][key]
            except KeyError:
              pval=Theta

            if ((pval == 1 or pval == 0)):
                flag=1
                break
            else:
                val+=pval
                count+=1

        if (flag == 1):
            if (pval==1 and y.iloc[i] == targetclass):
                TP+=1
            elif (pval==1 and y.iloc[i] != targetclass):
                FN+=1
            elif (pval==0 and y.iloc[i] != targetclass):
                TN+=1
            else:
                FP+=1
        else:
            Aval=round((val/count),3)
            if (Aval>=Theta and y.iloc[i] == targetclass):
                TP+=1
            elif (Aval<Theta and y.iloc[i] != targetclass):
                TN+=1
            elif (Aval>=Theta and y.iloc[i] != targetclass):
                FN+=1
            else:
                FP+=1
    try:
      Recall=round(TP/(TP+FN),4)
    except:
      Recall=0
    try:
      FPR=round(FP/(TN+FP),4)
    except:
      FPR=0
    try:
      Precision=round(TP/(TP+FP),4)
    except:
      Precision=0
    try:
      Accuracy=round((TP+TN)/(TP+TN+FP+FN),4)
    except:
      Accuracy=0
    try:
      FScore=round(2*Recall*Precision/(Recall+Precision),4)
    except:
      FScore=0
    print("TP=",TP,"\tTN=",TN,"\tFP=",FP,"\tFN=",FN)
    return Recall, FPR, Precision, FScore, Accuracy

In [17]:
Result=RST_Validation(X_train,y_train,pur,Th)
print(Result)

TP= 6927 	TN= 249834 	FP= 1242 	FN= 2338
(0.7477, 0.0049, 0.848, 0.7947, 0.9862)


In [18]:
ResultTest = RST_Validation(X_test,y_test,pur,Th)
print(ResultTest)

TP= 2384 	TN= 83211 	FP= 416 	FN= 770
(0.7559, 0.005, 0.8514, 0.8008, 0.9863)


In [ ]:
def CountPatterns(pur):
  value=0
  for outer_key, outer_value in pur.items():
    acount=0
    ncount=0
    for inner_key, inner_value in outer_value.items():
      if (inner_value == 0):
        acount=acount+1
      if (inner_value == 1):
        ncount=ncount+1
    print(f"{outer_key}: {acount}, {ncount}, {len(outer_value.items())}")

In [20]:
CountPatterns(pur)

Flow Duration: 11, 0, 12
Total Fwd Packets: 72, 0, 74
Total Backward Packets: 44, 0, 46
Fwd Packets Length Total: 9, 0, 10
Bwd Packets Length Total: 36, 0, 38
Fwd Packet Length Max: 35, 0, 41
Fwd Packet Length Mean: 38, 0, 47
Fwd Packet Length Std: 38, 0, 45
Bwd Packet Length Max: 13, 2, 29
Bwd Packet Length Mean: 4, 0, 15
Bwd Packet Length Std: 6, 1, 32
Flow Bytes/s: 79, 0, 94
Flow Packets/s: 14, 0, 36
Flow IAT Mean: 5, 0, 6
Flow IAT Std: 10, 0, 11
Flow IAT Max: 12, 0, 13
Flow IAT Min: 9, 0, 10
Fwd IAT Total: 11, 0, 12
Fwd IAT Mean: 5, 0, 6
Fwd IAT Std: 10, 0, 11
Fwd IAT Max: 12, 0, 13
Fwd IAT Min: 9, 0, 10
Bwd IAT Total: 7, 0, 101
Bwd IAT Mean: 40, 0, 101
Bwd IAT Std: 10, 0, 101
Bwd IAT Max: 7, 0, 101
Bwd IAT Min: 75, 0, 101
Fwd Header Length: 86, 0, 87
Bwd Header Length: 7, 0, 8
Fwd Packets/s: 16, 0, 44
Bwd Packets/s: 13, 0, 33
Packet Length Max: 33, 1, 49
Packet Length Mean: 12, 0, 22
Packet Length Std: 13, 0, 32
Packet Length Variance: 10, 0, 14
Avg Packet Size: 12, 0, 25
Avg Fwd 